# 33b — ConvNeXt-Tiny, fine-tuned

**One arm, one notebook.** Run this while your teammates run the other two; nothing here
collides — separate W&B run names, separate checkpoint folders, separate results files.

**The direct pairing with our own work.** `v27-convnext_style` — the same design,
written by us, trained from scratch at **414k parameters** — scored **0.8883**. This is
ConvNeXt pretrained on ImageNet at **28.0M**: 68x the parameters, plus a lot of GPU time
we did not spend.

If it does not clearly win, that is the strongest slide in the presentation. It says the
architecture was never the bottleneck.

Everything except the backbone is the recipe that won v30: our own 256-unit MLP head,
encoder at **1e-5** while the head runs at **7e-4**, cosine warmup, rotation p=0.5,
inverse-sqrt sampler, 128x128 one-hot, batch 256, patience 10.

**28.0M parameters, roughly ~2.5 h on an L4.**

## What you are comparing against

| run | | macro-F1 | Scratch |
|---|---|---|---|
| `dilated-style-64-dihedral8` | ours, 298k | 0.8995 | 0.793 |
| `v30-finetune_encoder_1e-5` | resnet18 fine-tuned | 0.8938 | 0.813 |
| `v27-resnet_style` | ours, 2.83M | 0.8900 | 0.759 |
| `v27-convnext_style` | ours, 414k | 0.8883 | 0.736 |
| `v27-baseline_cnn` | ours, 157k | 0.8646 | 0.715 |
| `v29-resnet18_frozen_onehot` | resnet18 **frozen** | 0.7351 | 0.401 |

The noise floor is **0.02** — `baseline_cnn` on one fixed config scored 0.8800, 0.8696 and
0.8646 across three runs. Anything closer than that is a tie, and a tie is a real finding
here, not a failure.

## Before you start

1. `git pull` in `/content/fdl-project`, then **Runtime > Restart session**. Colab caches
   `fdl_project` after the first import, so a pull alone will not take effect and you will
   get `Unknown model 'frozen_backbone_mlp'`.
2. `wandb login` in a terminal, or put `WANDB_KEY` in Colab Secrets.
3. Run every cell in order.

## 0. Colab web UI only — clone and authenticate

Skip if `/content/fdl-project` already exists.

In [ ]:
from getpass import getpass
from pathlib import Path
import subprocess

TARGET = Path("/content/fdl-project")
BRANCH = "feature/phase3-architectures"
REMOTE = "github.com/ezero3/fdl-project.git"


def run(*command: str) -> None:
    subprocess.run(command, check=True)


if TARGET.exists():
    print(f"{TARGET} already present -- pulling")
    run("git", "-C", str(TARGET), "fetch", "origin", BRANCH)
    run("git", "-C", str(TARGET), "checkout", BRANCH)
    run("git", "-C", str(TARGET), "pull", "--ff-only")
else:
    # Private repo, so the clone needs a personal access token. getpass keeps it
    # out of the notebook and out of the output.
    token = getpass("GitHub personal access token (input hidden): ").strip()
    run("git", "clone", "--branch", BRANCH,
        f"https://{token}@{REMOTE}", str(TARGET))
    # Drop the token from the stored remote; a later pull will ask again rather
    # than leaving a credential sitting in .git/config.
    run("git", "-C", str(TARGET), "remote", "set-url", "origin", f"https://{REMOTE}")
    del token

print(subprocess.run(["git", "-C", str(TARGET), "log", "--oneline", "-1"],
                     capture_output=True, text=True).stdout.strip())

## 1. Setup

Mounts Drive, copies the dataset, installs the package.

In [ ]:
import os, shutil, subprocess, sys
from pathlib import Path


def looks_like_the_repository(path: Path) -> bool:
    return (path / "pyproject.toml").exists() and (path / "src" / "fdl_project").is_dir()


REPO = next(
    (p for p in [Path.cwd(), *Path.cwd().parents, Path("/content/fdl-project")]
     if looks_like_the_repository(p)),
    None,
)
assert REPO is not None, "clone the repo to /content/fdl-project first"
os.chdir(REPO)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--ignore-requires-python",
                "-e", str(REPO), "--no-deps"], check=True)
source = str(REPO / "src")
if source not in sys.path:
    sys.path.insert(0, source)

import torch

DRIVE_ROOT = Path("/content/drive/MyDrive")
DRIVE = DRIVE_ROOT / "BICOCCA/FDL"
DATASET = REPO / "data/MIR-WM811K/WM811K.pkl"
EXPECTED_BYTES = 2_022_961_642


def mount_drive() -> bool:
    if DRIVE_ROOT.is_dir():
        return True
    try:
        from google.colab import drive

        drive.mount("/content/drive")   # idempotent; never force_remount
    except Exception as error:
        print(f"  Drive unavailable ({type(error).__name__})")
        return False
    return DRIVE_ROOT.is_dir()


HAS_DRIVE = mount_drive()
if not DATASET.exists():
    DATASET.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(DRIVE / "DATA/data/MIR-WM811K/WM811K.pkl", DATASET)
assert DATASET.stat().st_size == EXPECTED_BYTES, "wrong pickle: splits are row indices"

CHECKPOINTS = DRIVE / "checkpoints"
if HAS_DRIVE:
    CHECKPOINTS.mkdir(parents=True, exist_ok=True)

print(f"gpu     {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE'}")
print(f"drive   {'mounted' if HAS_DRIVE else 'NOT mounted'}")
print(f"dataset {DATASET.stat().st_size / 1024**3:.2f} GiB")

## 2. W&B

In [ ]:
USE_WANDB = True
WANDB_PROJECT = "wm811k-wafer-defects"

if USE_WANDB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "wandb"], check=True)
    import wandb

    if not wandb.api.api_key:
        # Colab Secrets work on the web UI. They time out under the VS Code
        # runtime, which is why the other notebooks tell you to use a terminal.
        # Add WANDB_KEY at the key icon in the left sidebar and enable it here.
        try:
            from google.colab import userdata

            wandb.login(key=userdata.get("WANDB_KEY"))
        except Exception as error:
            print(f"  Colab Secrets unavailable ({type(error).__name__})")

    if not wandb.api.api_key:
        USE_WANDB = False
        print("  not authenticated -- add WANDB_KEY to Colab Secrets, or run "
              "`wandb login` in a terminal, then rerun this cell")
    else:
        print(f"  wandb ready, project {WANDB_PROJECT!r}")


## 3. Train

One config: `configs/train/v33_modern_pretrained/19_convnext_tiny_finetune.yaml`.

The cell prints the parameter groups the optimizer built before training starts — the
encoder should be the large number at lr 1e-05 and the head the small one at 0.0007. If
they are the wrong way round, stop; something is misconfigured and the run is wasted.

Checkpoints go to Drive under this arm's own name, so if the runtime drops, rerunning this
cell resumes from the last epoch rather than restarting.

In [ ]:
import shutil, time

import pandas as pd

from fdl_project.config.loader import load_experiment_config
from fdl_project.config.registry import build_model
from fdl_project.data.datasets import load_wm811k_dataframe
from fdl_project.models.baseline_cnn import count_trainable_parameters
from fdl_project.training.optim import build_optimizer
from fdl_project.training.runner import run_experiment

SERIES = "v33_modern_pretrained"
CONFIG = REPO / "configs/train" / SERIES / "19_convnext_tiny_finetune.yaml"
assert CONFIG.exists(), f"{CONFIG} missing -- git pull, then Runtime > Restart session"

OUTPUT = REPO / "output" / SERIES
OUTPUT.mkdir(parents=True, exist_ok=True)

OVERRIDES = ["data.transform_device=cuda"]
if HAS_DRIVE:
    OVERRIDES.append(f"checkpoint.directory={CHECKPOINTS}")
if USE_WANDB:
    OVERRIDES += ["logging.wandb.enabled=true",
                  f"logging.wandb.project={WANDB_PROJECT}",
                  f"logging.wandb.tags=[{SERIES},finetune]"]

config = load_experiment_config(CONFIG, overrides=OVERRIDES)
assert config.data.augmentation.name == "rotation", "settled pipeline is rotation"
assert config.model.kwargs["freeze_encoder"] is False, "this arm fine-tunes"

model = build_model(config.model.name, **config.model.kwargs)
required = model.required_input_size
size = tuple(config.data.preprocessing.target_size)
assert required is None or size == (required, required), (
    f"{config.model.kwargs['architecture']} requires {required}px, config says {size}"
)
optimizer = build_optimizer(model, config.optimizer)
print(f"=== {config.name}  ({config.model.kwargs['architecture']}, {size[0]}px)")
for group in optimizer.param_groups:
    print(f"    {str(group.get('name')):10} lr={group['lr']:<9g} "
          f"{sum(p.numel() for p in group['params']):>12,} parameters")
print(f"    {sum(p.numel() for p in model.parameters()):,} total, "
      f"max_epochs {config.trainer.max_epochs}, patience "
      f"{config.trainer.early_stopping.patience}")
del model, optimizer

dataframe = load_wm811k_dataframe(DATASET)
started = time.monotonic()
result = run_experiment(config, overwrite=True, dataframe=dataframe)

macro = result.bootstrap.aggregate.set_index("metric").loc["macro_f1"]
per_class = result.validation.per_class_metrics.set_index("class_name")["f1"]
row = {
    "run": config.name,
    "backbone": config.model.kwargs["architecture"],
    "px": size[0],
    "macro_f1": round(float(macro.point_estimate), 4),
    "ci_lower": round(float(macro.ci_lower), 4),
    "ci_upper": round(float(macro.ci_upper), 4),
    "scratch_f1": round(float(per_class["Scratch"]), 3),
    "near_full_f1": round(float(per_class["Near-full"]), 3),
    "best_epoch": result.fit.best_epoch,
    "epochs": len(result.fit.history),
    "minutes": round((time.monotonic() - started) / 60, 1),
}
# One file per arm, so three notebooks running at once never overwrite each other.
frame = pd.DataFrame([row])
csv = OUTPUT / "convnext_tiny_results.csv"
frame.to_csv(csv, index=False)
if HAS_DRIVE:
    shutil.copy2(csv, DRIVE / f"{SERIES}_convnext_tiny.csv")

print(f"\n  macro-F1 {row['macro_f1']:.4f} [{row['ci_lower']:.4f}, {row['ci_upper']:.4f}]")
print(f"  Scratch {row['scratch_f1']:.3f}   Near-full {row['near_full_f1']:.3f}")
print(f"  best epoch {row['best_epoch']}/{row['epochs']}   {row['minutes']:.1f} min")
if row["best_epoch"] >= row["epochs"] - 2:
    print("  STILL IMPROVING AT THE CAP -- this number is a floor, say so when reporting it.")
print(f"  saved to {csv}")

## 4. Where it lands

Run this after section 3 finishes.

In [ ]:
# Everything below was measured on the same splits, same augmentation, same sampler.
REFERENCE = [
    ("dilated-style-64-dihedral8", "ours, 298k, old aug", 0.8995, 0.793),
    ("v30-finetune_encoder_1e-5",  "resnet18 fine-tuned", 0.8938, 0.813),
    ("v27-resnet_style",           "ours, 2.83M",         0.8900, 0.759),
    ("v27-convnext_style",         "ours, 414k",          0.8883, 0.736),
    ("v27-baseline_cnn",           "ours, 157k",          0.8646, 0.715),
    ("v29-resnet18_frozen_onehot", "resnet18 FROZEN",     0.7351, 0.401),
]
NOISE_FLOOR = 0.02   # baseline_cnn on one fixed config: 0.8800 / 0.8696 / 0.8646

table = pd.DataFrame(REFERENCE, columns=["run", "what", "macro_f1", "scratch_f1"])
mine = pd.DataFrame([{"run": row["run"], "what": "THIS NOTEBOOK",
                      "macro_f1": row["macro_f1"], "scratch_f1": row["scratch_f1"]}])
combined = pd.concat([mine, table]).sort_values("macro_f1", ascending=False)
pd.set_option("display.width", 200)
display(combined.reset_index(drop=True))

best_other = table["macro_f1"].max()
delta = row["macro_f1"] - best_other
verdict = "REAL" if abs(delta) > NOISE_FLOOR else "inside the noise floor"
print(f"vs the best measured so far ({best_other:.4f}): {delta:+.4f}  [{verdict}]")
print(f"vs our 414k convnext_style (0.8883):        {row['macro_f1'] - 0.8883:+.4f}")
print("\nA tie is a result here. Six runs now sit within 0.02 of each other across")
print("157k to 28M parameters, from-scratch and pretrained -- that is the finding.")

## 5. Report back

Paste the printed line into the group chat:

```
convnext_tiny: macro-F1 X.XXXX [lo, hi], Scratch X.XXX, best epoch N/M, T min
```

Flag it explicitly **if it says "STILL IMPROVING AT THE CAP"** — that means the number is a
floor and the arm needs a higher `trainer.max_epochs`, not that the backbone is worse.

The CSV is at `output/v33_modern_pretrained/convnext_tiny_results.csv` and is copied to
Drive automatically.